<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks-EN/1.3-mfcc-speech-recognition-bases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The importance of using MFCCs and the Discrete Cosine Transform (DCT)

Since we are using the RAVDESS dataset, part of our final goal is to differentiate the emotion with which a phrase was spoken using *Deep Learning* and *Machine Learning* techniques. This will allow us to obtain a series of results to compare both methods, contrasting their accuracy, computational cost, and performance.

But why is it essential to focus on MFCCs (*Mel-Frequency Cepstral Coefficients*)?

The answer is that the information in the cepstral coefficients can separate the excitation signal from the physical characteristics that filter it (source-filter model). For example, when the same note is played by two different instruments, the frequency components (harmonics) can be very similar, but the distribution of their intensity will not be identical. This is because the body of the instrument (or the human vocal tract) has unique physical properties that act as a filter, defining the timbre that finally reaches our ears. By isolating this "filter", we obtain a nearly pure description of **timbre**, regardless of the note or pitch being produced.

### Why is the DCT used?

It represents the final step in MFCC extraction and is performed for two fundamental technical reasons:

1. **Decorrelation (reducing redundancy):**
In a Mel spectrogram, the energies of adjacent frequency bands are highly correlated (if the 500 Hz band has high energy, the 510 Hz band likely does as well). This redundancy is inefficient for statistical modeling. The DCT takes this "overlapped" information and transforms it into independent or decorrelated coefficients. For a classification model, it is much more efficient and effective to process a vector of 13 to 20 independent coefficients than to analyze a dense matrix with redundant data.

2. **Information compression (envelope modeling):**
The **first coefficients** of the DCT capture the slow, general variations of the spectrum (the spectral envelope or timbre), while the **higher coefficients** represent rapid variations (fine details or noise). By applying the DCT, we can discard the highest coefficients and retain only the first ones (generally between 12 and 20), which concentrate all the acoustic information relevant to characterizing the emotional timbre of the voice.

![Pipeline MFCC](https://github.com/AcSsalazar/the-color-of-emotions/blob/main/Color-online-The-MFCC-feature-extraction-process.jpg?raw=1)

In [ ]:
import librosa
import matplotlib.pyplot as plt
import os
import sys
import numpy as np
import IPython.display as ipd

For this notebook, we will use two audio clips different from the ones used previously; this makes it easier to distinguish them by ear and to understand the differences between their characteristics. Below, you will be able to play those clips and confirm that the emotions contrast strongly.

In [ ]:
# Detect whether we are running the code in Google Colab

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


    BASE_DIR = "/content/drive/MyDrive/Actor_01"

else:

    BASE_DIR = "./Actor_01"

# From here on, we load the file the same way in both environments
angry_clip = os.path.join(BASE_DIR, "Angry.wav")
calm_clip = os.path.join(BASE_DIR, "Calm.wav")

print(f"Loading files from: {BASE_DIR}")

In [ ]:
ipd.Audio(angry_clip)

In [ ]:
ipd.Audio(calm_clip)

In [ ]:

def comparar_mfcc(file_1, file_2, label_1="Furioso", label_2="Calmado"):

    # 1. Load audios (same sr)
    y1, sr = librosa.load(file_1)
    y2, sr = librosa.load(file_2)

    # 2. Extract MFCCs (default is 20, we keep 13)
    mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=13)
    mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=13)

    # 3. Average over time to get a unique "profile" of the clip
    # (MFCC returns a 13 x frames matrix; we average the frames)
    mfcc1_avg = np.mean(mfcc1, axis=1)
    mfcc2_avg = np.mean(mfcc2, axis=1)


    # 4. Plot the comparison
    plt.figure(figsize=(12, 6))

    x = np.arange(1, 14) # Axis for the 13 coefficients

    plt.plot(x, mfcc1_avg, label=label_1, marker='o', linewidth=2)
    plt.plot(x, mfcc2_avg, label=label_2, marker='s', linewidth=2)

    plt.title('Timbre profile: MFCC coefficient comparison')
    plt.xlabel('MFCC coefficient index')
    plt.ylabel('Coefficient value (log-energy)')
    plt.xticks(x)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()


comparar_mfcc(calm_clip, angry_clip)

In [ ]:
y1, sr = librosa.load(angry_clip)
y2, sr = librosa.load(calm_clip)


mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=13)
mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=13)


plt.figure(figsize=(18, 5))
librosa.display.specshow(mfcc1,
                         x_axis="time",
                         sr=sr)
plt.colorbar(format="%+2.f")
plt.show()

In [ ]:
plt.figure(figsize=(18, 5))
librosa.display.specshow(mfcc2,
                         x_axis="time",
                         sr=sr)
plt.colorbar(format="%+2.f")
plt.show()

## The Delta-Delta

To understand Delta and Delta-Delta (second order), we must stop seeing MFCCs as a static photo and start seeing them as a video.

If the MFCCs are the "position" of the spectral envelope at a given moment, the deltas are its movement.

<br>

**1. The Concept of Derivative**

* In physics, if you have position, the first derivative is velocity and the second is acceleration. In audio it works exactly the same:

* Static MFCCs: They represent the "what" is being said (the timbre at a specific millisecond).

* **Delta (1st order)**: Represents the rate of change of the coefficients. It tells us how quickly the vocal tract is moving from one shape to another (for example, when moving from the letter "S" to "O").


* **Delta-Delta (2nd order)**: Represents acceleration. It tells us whether that sound change is slowing down or speeding up.

<br>

**2. Why is the 2nd order vital for emotions?**

Imagine someone who is angry versus someone who is sad:

* In anger: Mouth movements are abrupt and fast. MFCC coefficients change violently. Here, the Delta-Delta will have very high values (a lot of acceleration/braking in timbre).

* In sadness: Changes are slow and drawn out. Transitions between sounds are smooth. The Delta-Delta will be very low or close to zero, because there are no sudden "starts" in the voice.




Without the second order, the Machine Learning model would only see "photos" of timbre, but would miss the dynamics of speech, which is where emotion really lives.

In [ ]:
plt.figure(figsize=(14, 16))
# Deltas de orden 1
delta_011 = librosa.feature.delta(mfcc1, order=1)
delta_012 = librosa.feature.delta(mfcc2, order=1)
# Deltas de orden 2
delta_021 = librosa.feature.delta(mfcc1, order=2)
delta_022 = librosa.feature.delta(mfcc2, order=2)

def plot_delta(delta_array, sr, title, subplot_pos):
    plt.subplot(subplot_pos[0], subplot_pos[1], subplot_pos[2])
    librosa.display.specshow(delta_array,
                             x_axis="time",
                             sr=sr)
    plt.colorbar(format="%+2.f")
    plt.title(title)

plot_delta(delta_011, sr, 'Delta MFCC (Furioso Clip)', (4, 1, 1)) # 4 Filas, 1 Columna, 1st plot
plot_delta(delta_012, sr, 'Delta MFCC (Calmao Clip)', (4, 1, 2))  # 4 Filas, 1 Columna, 2st plot
plot_delta(delta_021, sr, 'Delta-Delta MFCC (Furioso Clip)', (4, 1, 3)) # 4 Filas, 1 Columna, 3st plot
plot_delta(delta_022, sr, 'Delta-Delta MFCC (Calmado Clip)', (4, 1, 4))  # 4 Filas, 1 Columna, 4st plot

plt.tight_layout()
plt.show()